# 03 - Stationarity and Cointegration

This notebook applies ADF and KPSS tests to price levels, first differences, and returns, then tests for long-run equilibrium relationships using Johansen cointegration.

In [1]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR, RESULTS_DIR, SERIES_FILES
from src.utils import configure_logging, ensure_directory

configure_logging()
sns.set_theme(style="whitegrid")
ensure_directory(PROCESSED_DATA_DIR)
ensure_directory(RESULTS_DIR)
ensure_directory(FIGURES_DIR)

from src.cointegration import interpret_johansen_results, johansen_cointegration_test
from src.stationarity import stationarity_summary

## Load Processed Data

VAR models require stationary inputs unless the analysis is extended to a cointegrated VECM framework.

In [2]:
clean = pd.read_csv(PROCESSED_DATA_DIR / "market_data_clean.csv", parse_dates=["Date"], index_col="Date")
first_difference = pd.read_csv(PROCESSED_DATA_DIR / "market_data_first_difference.csv", parse_dates=["Date"], index_col="Date")
returns = pd.read_csv(PROCESSED_DATA_DIR / "market_data_returns.csv", parse_dates=["Date"], index_col="Date")

clean.head()

,MASI,CAC40,SP500,EUR_MAD,USD_MAD
Date,,,,,
2015-01-02,9643.190430,4252.290039,2058.199951,10.9523,9.1250
2015-01-06,9651.780273,4083.500000,2002.599976,10.9534,9.2130
2015-01-07,9750.299805,4112.729980,2025.900024,10.9345,9.2360
2015-01-08,9803.870117,4260.189941,2062.100098,10.9251,9.2645
2015-01-09,9993.190430,4179.069824,2044.800049,10.9297,9.2300


## ADF and KPSS Tests

ADF has a unit-root null hypothesis, while KPSS has a stationarity null hypothesis. Reading both together gives a more balanced decision.

In [3]:
stationarity_prices = stationarity_summary(clean)
stationarity_diff = stationarity_summary(first_difference)
stationarity_returns = stationarity_summary(returns)

stationarity_prices.to_csv(RESULTS_DIR / "stationarity_raw_prices.csv", index=False)
stationarity_diff.to_csv(RESULTS_DIR / "stationarity_first_difference.csv", index=False)
stationarity_returns.to_csv(RESULTS_DIR / "stationarity_returns.csv", index=False)

stationarity_prices

c:\Users\SYB DELL\Desktop\market-causality-analysis\src\stationarity.py:29: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  statistic, p_value, used_lags, critical_values = kpss(clean, regression="c", nlags="auto")
c:\Users\SYB DELL\Desktop\market-causality-analysis\src\stationarity.py:29: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  statistic, p_value, used_lags, critical_values = kpss(clean, regression="c", nlags="auto")
c:\Users\SYB DELL\Desktop\market-causality-analysis\src\stationarity.py:29: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  statistic, p_value, used_lags, critical_values = kpss(clean, regression="c", nlags=

,variable,ADF statistic,ADF p-value,ADF conclusion,KPSS statistic,KPSS p-value,KPSS conclusion,final conclusion
0,MASI,-0.351448,0.917895,Non-stationary,4.590180,0.010000,Non-stationary,Likely non-stationary or mixed evidence
1,CAC40,-1.173601,0.684847,Non-stationary,7.652641,0.010000,Non-stationary,Likely non-stationary or mixed evidence
2,SP500,1.168365,0.995760,Non-stationary,7.815362,0.010000,Non-stationary,Likely non-stationary or mixed evidence
3,EUR_MAD,-3.211242,0.019343,Stationary,1.576754,0.010000,Non-stationary,Likely non-stationary or mixed evidence
4,USD_MAD,-2.327252,0.163329,Non-stationary,0.439704,0.060042,Stationary,Likely non-stationary or mixed evidence


In [4]:
stationarity_diff

,variable,ADF statistic,ADF p-value,ADF conclusion,KPSS statistic,KPSS p-value,KPSS conclusion,final conclusion
0,MASI,-24.498644,0.000000e+00,Stationary,0.160046,0.1,Stationary,Likely stationary
1,CAC40,-52.224902,0.000000e+00,Stationary,0.024622,0.1,Stationary,Likely stationary
2,SP500,-12.217493,1.126870e-22,Stationary,0.281310,0.1,Stationary,Likely stationary
3,EUR_MAD,-23.407683,0.000000e+00,Stationary,0.022336,0.1,Stationary,Likely stationary
4,USD_MAD,-14.903580,1.494412e-27,Stationary,0.106209,0.1,Stationary,Likely stationary


In [5]:
stationarity_returns

,variable,ADF statistic,ADF p-value,ADF conclusion,KPSS statistic,KPSS p-value,KPSS conclusion,final conclusion
0,MASI,-33.258764,0.000000e+00,Stationary,0.129831,0.1,Stationary,Likely stationary
1,CAC40,-17.853432,3.073107e-30,Stationary,0.015669,0.1,Stationary,Likely stationary
2,SP500,-14.935373,1.347069e-27,Stationary,0.068477,0.1,Stationary,Likely stationary
3,EUR_MAD,-23.445676,0.000000e+00,Stationary,0.023026,0.1,Stationary,Likely stationary
4,USD_MAD,-25.759404,0.000000e+00,Stationary,0.112459,0.1,Stationary,Likely stationary


## Johansen Cointegration Test

Cointegration tests whether non-stationary price series share one or more stable long-run relationships. Evidence of cointegration would motivate a VECM extension.

In [6]:
johansen_results = johansen_cointegration_test(clean, det_order=0, k_ar_diff=1)
johansen_results.to_csv(RESULTS_DIR / "johansen_cointegration_prices.csv", index=False)
interpretation = interpret_johansen_results(johansen_results)

print(interpretation)
johansen_results

At the 5% level, the trace test rejects cointegration ranks up to 0. This suggests at least 1 cointegrating relationship(s), subject to lag and deterministic-term choices.


,rank_null,trace_statistic,critical_value_90,critical_value_95,critical_value_99,reject_at_5_percent
0,0,73.783669,65.8202,69.8189,77.8202,True
1,1,39.174766,44.4929,47.8545,54.6815,False
2,2,18.839454,27.0669,29.7961,35.4628,False
3,3,6.222040,13.4294,15.4943,19.9349,False
4,4,1.414930,2.7055,3.8415,6.6349,False


## Interpretation

Price levels are commonly non-stationary because stock indices and exchange rates can trend over time. First differences and returns often become stationary, making them suitable for VAR and Granger causality. If Johansen results indicate cointegration, the price series may share a long-term equilibrium relation, but the short-run VAR in this project remains focused on stationary transformed data.